## Build Camera–Species Agreement Dataset

This notebook creates a camera–species response dataset comparing: 

- Species **observed by Snapshot USA** at each camera location.
- Species **predicted by IUCN range maps** within each camera's 1-km footprint.

The final modeling dataset will identify, for each species predicted by IUCN at a camera location, whether that species was observed by Snapshot USA.

In [184]:
from pathlib import Path

import pandas as pd
import geopandas as gpd
import numpy as np

In [185]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


In [186]:
# --------------------------------------------------
# Define project directories and file paths
# --------------------------------------------------

CLEANED_PATH = "../cleaned"
PREPROCESSED_PATH = "../preprocessed_data"
OUTPUT_PATH = "../../outputs/species_level_analysis"

# Input files
SSUSA_FILE = f"{CLEANED_PATH}/ssusa_cleaned.csv"
IUCN_FILE = f"{CLEANED_PATH}/iucn_cleaned.shp"

CAMERA_FOOTPRINTS_FILE = (
    f"{PREPROCESSED_PATH}/ssusa_camera_footprints_1km.geojson"
)

# Output file
CAMERA_SPECIES_FILE = (
    f"{OUTPUT_PATH}/camera_species_agreement.csv"
)

# Create output directory if needed
Path(OUTPUT_PATH).mkdir(
    parents=True,
    exist_ok=True,
)

In [187]:
# Verify file locations
# ==========================================================

print("Input files")
print(f"Snapshot USA      : {SSUSA_FILE}")
print(f"IUCN ranges       : {IUCN_FILE}")
print(f"Camera footprints : {CAMERA_FOOTPRINTS_FILE}")

print("\nOutput")
print(f"Camera-species dataset : {CAMERA_SPECIES_FILE}")

Input files
Snapshot USA      : ../cleaned/ssusa_cleaned.csv
IUCN ranges       : ../cleaned/iucn_cleaned.shp
Camera footprints : ../preprocessed_data/ssusa_camera_footprints_1km.geojson

Output
Camera-species dataset : ../../outputs/species_level_analysis/camera_species_agreement.csv


### Load and Inspect the Input Datasets

In [188]:
# Load datasets
# --------------------------------------------------
# Snapshot USA observations
ssusa_all = pd.read_csv(
    SSUSA_FILE,
    low_memory=False,
)

# Camera footprints and IUCN range polygons
camera_footprints = gpd.read_file(
    CAMERA_FOOTPRINTS_FILE
)

iucn = gpd.read_file(
    IUCN_FILE
)

print("Datasets loaded successfully.\n")

Datasets loaded successfully.



In [189]:
# Dataset dimensions
# --------------------------------------------------
print(
    f"SSUSA observations : {ssusa_all.shape[0]:,} rows × "
    f"{ssusa_all.shape[1]} columns"
)

print(
    f"Camera footprints  : {camera_footprints.shape[0]:,} rows × "
    f"{camera_footprints.shape[1]} columns"
)

print(
    f"IUCN polygons      : {iucn.shape[0]:,} rows × "
    f"{iucn.shape[1]} columns"
)

SSUSA observations : 713,319 rows × 29 columns
Camera footprints  : 7,340 rows × 4 columns
IUCN polygons      : 751 rows × 28 columns


### Create a Common Camera Identifier

The Snapshot USA observation table does not contain a camera identifier. To link observations with the 1-km camera footprints, recreate the camera identifier from the camera longitude and latitude using the same function that was used when generating the camera footprints.

This identifier will be used to join species observations with camera footprints throughout the analysis.

In [190]:
def make_camera_id(lon, lat):
    """Create a unique camera identifier from longitude and latitude."""
    return f"{lon:.8f}_{lat:.8f}"

ssusa_all["camera_fp_id"] = ssusa_all.apply(
    lambda row: make_camera_id(
        row["Longitude"],
        row["Latitude"]
    ),
    axis=1,
)

print(f"Created camera IDs for {len(ssusa_all):,} observations.")

Created camera IDs for 713,319 observations.


In [191]:
# Verify camera identifiers
# --------------------------------------------------

print(f"Unique cameras in SSUSA: {ssusa_all['camera_fp_id'].nunique():,}")

print(f"Camera footprints:       {camera_footprints['camera_fp_id'].nunique():,}")

matched = ssusa_all["camera_fp_id"].isin(
    camera_footprints["camera_fp_id"]
).sum()

print(f"Observations with matching camera footprint: {matched:,} / {len(ssusa_all):,}")

Unique cameras in SSUSA: 7,340
Camera footprints:       7,340
Observations with matching camera footprint: 713,319 / 713,319


### Filter Species by Body-Mass Threshold

Retain only species that meet the selected body-mass threshold in both datasets.

- **Snapshot USA:** `Above_Threshold == True`
- **IUCN:** `ab_thres == True`

In [192]:
# create a working copy 
ssusa = ssusa_all.copy()

In [193]:
print("Before filtering")
print(f"SSUSA observations : {len(ssusa):,}")
print(f"Unique cameras in SSUSA: {ssusa['camera_fp_id'].nunique():,}")
print(f"IUCN polygons      : {len(iucn):,}")

# Snapshot USA
ssusa = ssusa[ssusa["Above_Threshold"]].copy()

# IUCN
iucn = iucn[iucn["ab_thres"]].copy()

print("\nAfter filtering")
print(f"SSUSA observations : {len(ssusa):,}")
print(f"Unique cameras in SSUSA: {ssusa['camera_fp_id'].nunique():,}")
print(f"IUCN polygons      : {len(iucn):,}")

print("\nUnique species retained")
print(f"SSUSA : {ssusa['Species_Name'].nunique()}")
print(f"IUCN  : {iucn['sci_name'].nunique()}")



Before filtering
SSUSA observations : 713,319
Unique cameras in SSUSA: 7,340
IUCN polygons      : 751

After filtering
SSUSA observations : 673,198
Unique cameras in SSUSA: 7,323
IUCN polygons      : 200

Unique species retained
SSUSA : 58
IUCN  : 117


### Select the Required Variables

For this notebook we only need the fields required to determine:

- Which species were observed at each camera (Snapshot USA)
- Which species are predicted to occur at each camera (IUCN)|

In [194]:
# Snapshot USA observations
ssusa_species = ssusa[
    [
        "camera_fp_id",
        "Species_Name",
    ]
].copy()

# Camera footprints
camera_fp = camera_footprints[
    [
        "camera_fp_id",
        "geometry",
    ]
].copy()

# IUCN ranges
iucn_species = iucn[
    [
        "sci_name",
        "geometry",
    ]
].copy()

print(f"SSUSA observations: {len(ssusa_species):,}")
print(f"Camera footprints: {len(camera_fp):,}")
print(f"IUCN polygons: {len(iucn_species):,}")

SSUSA observations: 673,198
Camera footprints: 7,340
IUCN polygons: 200


In [196]:
# Create unique camera-species observations
# --------------------------------------------------

ssusa_species = (
    ssusa_species
    .dropna(subset=["camera_fp_id", "Species_Name"])
    .drop_duplicates(
        subset=[
            "camera_fp_id",
            "Species_Name",
        ]
    )
    .copy()
)

ssusa_species["ssusa_observed"] = 1

print(f"Unique camera-species pairs: {len(ssusa_species):,}")
print(f"Unique cameras:              {ssusa_species['camera_fp_id'].nunique():,}")
print(f"Unique SSUSA species:              {ssusa_species['Species_Name'].nunique():,}")

display(ssusa_species.head())

Unique camera-species pairs: 29,307
Unique cameras:              7,323
Unique SSUSA species:              58


,camera_fp_id,Species_Name,ssusa_observed
0,-136.22250000_59.42643000,ursus arctos,1
55,-136.22250000_59.42643000,alces alces,1
99,-135.92880000_59.39905000,canis latrans,1
101,-135.92880000_59.39905000,ursus arctos,1
103,-135.92880000_59.39905000,lynx canadensis,1


In [197]:
missing_cameras = sorted(
    set(camera_fp["camera_fp_id"]) -
    set(ssusa_species["camera_fp_id"])
)

print(f"Number of cameras removed after threshold filtering: {len(missing_cameras)}")

Number of cameras removed after threshold filtering: 17


In [198]:
ssusa_all[
    ssusa_all["camera_fp_id"].isin(missing_cameras)
][
    [
        "camera_fp_id",
        "Species_Name",
        "Above_Threshold"
    ]
].sort_values("camera_fp_id")





,camera_fp_id,Species_Name,Above_Threshold
88639,-108.02740000_47.93385000,neogale frenata,False
88640,-108.02740000_47.93385000,neogale frenata,False
673640,-119.86317000_39.35113000,tamiasciurus douglasii,False
673641,-119.86317000_39.35113000,tamiasciurus douglasii,False
195102,-122.95000000_47.96000000,tamiasciurus douglasii,False
...,...,...,...
92211,-79.74657000_39.65635000,tamias striatus,False
92212,-79.74657000_39.65635000,tamias striatus,False
92220,-79.74657000_39.65635000,tamias striatus,False
275441,-82.14459000_36.09301000,tamiasciurus hudsonicus,False


In [199]:
# print if any of the rows in missing cameras have Above_Threshold = True
print("Observations in missing cameras with Above_Threshold = True:")
print(
    ssusa_all[
        (ssusa_all["camera_fp_id"].isin(missing_cameras)) &
        (ssusa_all["Above_Threshold"])
    ][
        [
            "camera_fp_id",
            "Species_Name",
            "Above_Threshold"
        ]
    ].sort_values("camera_fp_id").shape[0]
)

Observations in missing cameras with Above_Threshold = True:
0


### Create Camera–Species IUCN Predictions

Identify all species whose IUCN range polygons intersect each 1-km camera footprint.

Each resulting camera–species pair represents a species that is predicted to occur at that camera according to the IUCN range maps.

The resulting table contains:

- `camera_fp_id`
- `sci_name`
- `iucn_predicted`

In [200]:
# Ensure both datasets use the same CRS
# --------------------------------------------------

if camera_fp.crs != iucn_species.crs:
    iucn_species = iucn_species.to_crs(camera_fp.crs)

print("Camera footprint CRS:", camera_fp.crs)
print("IUCN CRS:", iucn_species.crs)

Camera footprint CRS: EPSG:5070
IUCN CRS: EPSG:5070


In [201]:
# Spatial join: camera footprints × IUCN ranges
# --------------------------------------------------

camera_iucn = gpd.sjoin(
    camera_fp,
    iucn_species,
    how="inner",
    predicate="intersects",
)

In [135]:
# Keep unique camera-species predictions
# --------------------------------------------------
camera_iucn = (
    camera_iucn[
        [
            "camera_fp_id",
            "sci_name",
        ]
    ]
    .dropna(
        subset=[
            "camera_fp_id",
            "sci_name",
        ]
    )
    .drop_duplicates()
    .copy()
)

# Label each camera-species pair as predicted
camera_iucn["iucn_predicted"] = 1

In [202]:
print(f"Camera-species predictions: {len(camera_iucn):,}")
print(f"Unique cameras inside range: {camera_iucn['camera_fp_id'].nunique():,}")
print(f"Unique species: {camera_iucn['sci_name'].nunique():,}")

camera_iucn.head()

Camera-species predictions: 108,914
Unique cameras inside range: 7,223
Unique species: 64


,camera_fp_id,geometry,index_right,sci_name
0,-136.22250000_59.42643000,"POLYGON ((-2435854.994 4519384.214, -2435859.8...",569,odocoileus hemionus
0,-136.22250000_59.42643000,"POLYGON ((-2435854.994 4519384.214, -2435859.8...",667,oreamnos americanus
0,-136.22250000_59.42643000,"POLYGON ((-2435854.994 4519384.214, -2435859.8...",577,marmota caligata
0,-136.22250000_59.42643000,"POLYGON ((-2435854.994 4519384.214, -2435859.8...",564,ursus arctos
0,-136.22250000_59.42643000,"POLYGON ((-2435854.994 4519384.214, -2435859.8...",146,lynx canadensis


In [141]:
# Save camera-species prediction table
# --------------------------------------------------

IUCN_PREDICTIONS_FILE = f"{OUTPUT_PATH}/camera_species_iucn_predictions.csv"
    

camera_iucn.to_csv(
    IUCN_PREDICTIONS_FILE,
    index=False,
)

print(f"Saved camera-species predictions to:\n{IUCN_PREDICTIONS_FILE}")

Saved camera-species predictions to:
../../outputs/species_level_analysis/camera_species_iucn_predictions.csv


### Combine the IUCN predictions with the Snapshot USA observations.

In [142]:
# Standardize the species column name - Rename both columns to `species` so the datasets can be merged consistently.
# --------------------------------------------------

ssusa_species = ssusa_species.rename(
    columns={
        "Species_Name": "species"
    }
)

camera_iucn = camera_iucn.rename(
    columns={
        "sci_name": "species"
    }
)

print("Snapshot USA columns:")
print(ssusa_species.columns.tolist())

print("\nIUCN prediction columns:")
print(camera_iucn.columns.tolist())

Snapshot USA columns:
['camera_fp_id', 'species', 'ssusa_observed']

IUCN prediction columns:
['camera_fp_id', 'species', 'iucn_predicted']


In [147]:
# Create the complete analysis species list
# --------------------------------------------------

ssusa_species_set = set(
    ssusa_species["species"].dropna().unique()
)

iucn_species_set = set(
    camera_iucn["species"].dropna().unique()
)

analysis_species = sorted(
    ssusa_species_set.union(iucn_species_set)
)

print(f"Snapshot USA species: {len(ssusa_species_set):,}")
print(f"IUCN species: {len(iucn_species_set):,}")
print(f"Species in analysis: {len(analysis_species):,}")


Snapshot USA species: 58
IUCN species: 64
Species in analysis: 66


In [144]:
# Compare species represented in each dataset
# --------------------------------------------------

ssusa_only = sorted(
    ssusa_species_set - iucn_species_set
)

iucn_only = sorted(
    iucn_species_set - ssusa_species_set
)

print(f"Species only in Snapshot USA: {len(ssusa_only)}")
print(f"Species only in IUCN: {len(iucn_only)}")

print("\nSnapshot USA only:")
print(ssusa_only)

print("\nIUCN only:")
print(iucn_only)

Species only in Snapshot USA: 2
Species only in IUCN: 8

Snapshot USA only:
['bison bison', 'sus scrofa']

IUCN only:
['cynomys gunnisoni', 'leopardus pardalis', 'lepus europaeus', 'marmota caligata', 'marmota flaviventris', 'marmota olympus', 'oreamnos americanus', 'sylvilagus transitionalis']


In [148]:
# Create the Complete Camera–Species Grid

camera_ids = (
    camera_fp[["camera_fp_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# --------------------------------------------------
# Create a table of all analysis species
# --------------------------------------------------

species_ids = pd.DataFrame(
    {
        "species": analysis_species
    }
)

print(f"Unique cameras: {len(camera_ids):,}")
print(f"Analysis species: {len(species_ids):,}")

Unique cameras: 7,340
Analysis species: 66


In [57]:
camera_species_grid = camera_ids.merge(
    species_ids,
    how="cross",
)

print(
    "Total camera-species combinations:",
    f"{len(camera_species_grid):,}"
)

camera_species_grid.head()

Total camera-species combinations: 484,440


,camera_fp_id,species
0,-136.22250000_59.42643000,alces alces
1,-136.22250000_59.42643000,antilocapra americana
2,-136.22250000_59.42643000,aplodontia rufa
3,-136.22250000_59.42643000,bassariscus astutus
4,-136.22250000_59.42643000,bison bison


In [149]:
# Validate the expected number of rows
# --------------------------------------------------

expected_rows = (
    len(camera_ids) *
    len(species_ids)
)

print(f"Expected rows: {expected_rows:,}")
print(f"Actual rows:   {len(camera_species_grid):,}")

Expected rows: 484,440
Actual rows:   484,440


In [150]:
# Add Snapshot USA Observations
camera_species_agreement = camera_species_grid.merge(
    ssusa_species[
        [
            "camera_fp_id",
            "species",
            "ssusa_observed",
        ]
    ],
    on=[
        "camera_fp_id",
        "species",
    ],
    how="left",
)

# Species not observed at a camera are assigned 0
camera_species_agreement["ssusa_observed"] = (
    camera_species_agreement["ssusa_observed"]
    .fillna(0)
    .astype("int8")
)

In [151]:
# Summary of Snapshot USA observations
print(camera_species_agreement["ssusa_observed"].value_counts())
print("Total rows in camera_species_agreement", len(camera_species_agreement))

camera_species_agreement.head()

ssusa_observed
0    455133
1     29307
Name: count, dtype: int64
Total rows in camera_species_agreement 484440


,camera_fp_id,species,ssusa_observed
0,-136.22250000_59.42643000,alces alces,1
1,-136.22250000_59.42643000,antilocapra americana,0
2,-136.22250000_59.42643000,aplodontia rufa,0
3,-136.22250000_59.42643000,bassariscus astutus,0
4,-136.22250000_59.42643000,bison bison,0


In [152]:
# Add IUCN Predictions

camera_species_agreement = camera_species_agreement.merge(
    camera_iucn[
        [
            "camera_fp_id",
            "species",
            "iucn_predicted",
        ]
    ],
    on=[
        "camera_fp_id",
        "species",
    ],
    how="left",
)

# Species not predicted at a camera are assigned 0
camera_species_agreement["iucn_predicted"] = (
    camera_species_agreement["iucn_predicted"]
    .fillna(0)
    .astype("int8")
)

In [153]:
print(
    camera_species_agreement[
        "iucn_predicted"
    ].value_counts()
)

camera_species_agreement.head()

iucn_predicted
0    375527
1    108913
Name: count, dtype: int64


,camera_fp_id,species,ssusa_observed,iucn_predicted
0,-136.22250000_59.42643000,alces alces,1,1
1,-136.22250000_59.42643000,antilocapra americana,0,0
2,-136.22250000_59.42643000,aplodontia rufa,0,0
3,-136.22250000_59.42643000,bassariscus astutus,0,0
4,-136.22250000_59.42643000,bison bison,0,0


In [231]:
# save unfiltered camera_species_agreement to the csv file 
CAMERA_SPECIES_DETECTION = f"{OUTPUT_PATH}/camera_species_detection.csv"
    
camera_species_agreement.to_csv(
    CAMERA_SPECIES_DETECTION,
    index=False,
)

print(f"Saved to: {CAMERA_SPECIES_DETECTION}")

Saved to: ../../outputs/species_level_analysis/camera_species_detection.csv


### Restrict the Analysis to IUCN-Predicted Occurrences

The objective of this analysis is to understand why Snapshot USA does or does not observe species within the broad geographic ranges predicted by the IUCN.

Therefore, retain only camera–species pairs where the species is predicted to occur according to the IUCN range maps (`iucn_predicted = 1`).

The response variable is then:

- `ssusa_observed = 1` → species observed
- `ssusa_observed = 0` → species not observed

In [166]:
# Restrict to IUCN-predicted occurrences
# --------------------------------------------------
species_detection = (
    camera_species_agreement[
        camera_species_agreement["iucn_predicted"] == 1
    ]
    .copy()
)

print(f"Rows before IUCN range restricted: {len(camera_species_agreement):,}")

print(f"Rows retained: {len(species_detection):,}")




Rows before IUCN range restricted: 484,440
Rows retained: 108,913


In [203]:
print(f"Number of species before IUCN range restricted: {camera_species_agreement['species'].nunique()}")
print(f"Number of cameras before IUCN range restricted: {camera_species_agreement['camera_fp_id'].nunique()}")
print(f"Number of species: {species_detection['species'].nunique()}")
print(f"Number of cameras after IUCN range restricted: {species_detection['camera_fp_id'].nunique()}")

Number of species before IUCN range restricted: 66
Number of cameras before IUCN range restricted: 7340
Number of species: 64
Number of cameras after IUCN range restricted: 7223


In [170]:
species_before = set(camera_species_agreement["species"].dropna().unique())
species_after = set(species_detection["species"].dropna().unique())
species_removed = sorted(species_before - species_after)

print(species_removed)

['bison bison', 'sus scrofa']


In [ ]:
species_detection.head()

,camera_fp_id,species,ssusa_observed,iucn_predicted
0,-136.22250000_59.42643000,alces alces,1,1
5,-136.22250000_59.42643000,canis latrans,0,1
6,-136.22250000_59.42643000,canis lupus,0,1
15,-136.22250000_59.42643000,erethizon dorsatum,0,1
16,-136.22250000_59.42643000,gulo gulo,0,1


#### Species Observation Ratios IUCN Restricted 


In [221]:
species_summary = (
    species_detection
    .groupby("species")
    .agg(
        iucn_predicted_cameras=("iucn_predicted", "sum"),
        ssusa_observed_cameras=("ssusa_observed", "sum"),
    )
    .reset_index()
)

species_summary = species_summary.sort_values(
    "species",
    ascending=True,
)

species_summary.head(10)

,species,iucn_predicted_cameras,ssusa_observed_cameras
0,alces alces,834,116
1,antilocapra americana,630,119
2,aplodontia rufa,251,13
3,bassariscus astutus,1209,21
4,canis latrans,7101,3113
5,canis lupus,595,110
6,canis rufus,34,4
7,cervus elaphus,858,241
8,conepatus leuconotus,328,15
9,cynomys gunnisoni,26,0


In [222]:
species_rf_base = species_summary[
    [
        "species",
        "ssusa_observed_cameras",
        "iucn_predicted_cameras",
    ]
].copy()

In [223]:
print(f"Rows: {len(species_rf_base):,}")
print(f"Species: {species_rf_base['species'].nunique():,}")

print(
    species_rf_base["ssusa_observed_cameras"]
    .value_counts()
    .sort_index()
)

Rows: 64
Species: 64
ssusa_observed_cameras
0       9
1       1
2       1
4       2
5       3
10      1
12      1
13      2
15      2
16      1
17      1
21      2
23      1
25      1
26      1
28      1
29      1
35      1
45      1
51      1
62      1
69      1
82      1
95      1
108     1
110     1
116     2
118     1
119     1
126     1
137     1
138     1
178     1
185     1
241     1
519     1
662     1
755     1
796     1
839     1
853     1
983     1
1023    1
1103    1
1973    1
3088    1
3113    1
3610    1
5326    1
Name: count, dtype: int64


In [ ]:
SPECIES_RF_BASE_FILE = f"{OUTPUT_PATH}/camera_species_IUCN_Range_Restricted_rf_base.csv"
    
species_rf_base.to_csv(
    SPECIES_RF_BASE_FILE,
    index=False,
)

print(f"Saved to: {SPECIES_RF_BASE_FILE}")

Saved to: ../../outputs/species_level_analysis/camera_species_rf_base.csv


#### Species Observation Ratios not IUCN Restricted 



In [225]:
species_summary = (
    camera_species_agreement
    .groupby("species")
    .agg(
        iucn_predicted_cameras=("iucn_predicted", "sum"),
        ssusa_observed_cameras=("ssusa_observed", "sum"),
    )
    .reset_index()
)

species_summary = species_summary.sort_values(
    "species",
    ascending=True,
)

species_summary.head(10)

,species,iucn_predicted_cameras,ssusa_observed_cameras
0,alces alces,834,165
1,antilocapra americana,630,130
2,aplodontia rufa,251,13
3,bassariscus astutus,1209,21
4,bison bison,0,175
5,canis latrans,7101,3136
6,canis lupus,595,110
7,canis rufus,34,5
8,cervus elaphus,858,325
9,conepatus leuconotus,328,15


In [226]:
species_rf_base = species_summary[
    [
        "species",
        "ssusa_observed_cameras",
        "iucn_predicted_cameras",
    ]
].copy()

In [227]:
print(f"Rows: {len(species_rf_base):,}")
print(f"Species: {species_rf_base['species'].nunique():,}")

Rows: 66
Species: 66


In [229]:
SPECIES_RF_BASE_FILE = f"{OUTPUT_PATH}/camera_species_no_IUCN_restrication__rf_base.csv"
    
species_rf_base.to_csv(
    SPECIES_RF_BASE_FILE,
    index=False,
)

print(f"Saved to: {SPECIES_RF_BASE_FILE}")

Saved to: ../../outputs/species_level_analysis/camera_species_no_IUCN_restrication__rf_base.csv
